Ailene, This is the assignment from your video that you asked us to do. 

In [1]:
import requests
import pandas as pd
from datetime import datetime
import time

In [ ]:
API_KEY = # <your_api_key>
BASE_URL = "http://api.weatherapi.com/v1/"

In [19]:
def get_city_data(city, api_key):
    today = datetime.now().strftime("%Y-%m-%d")

    try:
        # --- Combined Weather Call ---
        forecast_url = f"{BASE_URL}forecast.json?key={api_key}&q={city}&days=3&aqi=yes"
        forecast_res = requests.get(forecast_url)
        forecast_res.raise_for_status()

        # --- Astronomy Call ---
        astro_url = f"{BASE_URL}astronomy.json?key={api_key}&q={city}&dt={today}"
        astro_res = requests.get(astro_url)
        astro_res.raise_for_status()

        # Parse JSON
        data = forecast_res.json()
        astro_data = astro_res.json()

        location = data['location']
        current = data['current']
        forecast_days = data['forecast']['forecastday']
        astro = astro_data['astronomy']['astro']

        # --- Build Summary ---
        city_summary = {
            "date_raw": datetime.now().strftime("%Y-%m-%d"),
            "date": datetime.now().strftime("%A, %B %#d %Y"),
            "city": location['name'],
            "region": location['region'],
            "country": location['country'],
            "temp_f": current['temp_f'],
            "feels_like": current['feelslike_f'],
            "condition": current['condition']['text'],
            "humidity": current['humidity'],
            "wind_mph": current['wind_mph'],
            "uv": current['uv'],
            "precip_in": current['precip_in'],
            "sunrise": astro['sunrise'],
            "sunset": astro['sunset'],
            "moon_phase": astro['moon_phase']
        }
        for i, day in enumerate(forecast_days):
            city_summary[f"date_{i+1}"] = day['date']
            city_summary[f"max_temp_day_{i+1}"] = day['day']['maxtemp_f']
            city_summary[f"min_temp_day_{i+1}"] = day['day']['mintemp_f']
            city_summary[f"avg_temp_day_{i+1}"] = day['day']['avgtemp_f']
        return city_summary

    except requests.exceptions.RequestException as e:
        print(f"Error fetching {city}: {e}")
        return None

In [20]:
cities = [
    "Louisville",
    "Lexington",
    "Cincinnati",
    "Paducah",
    "Bowling Green",
    "Owensboro",
    "Elizabethtown"
]

In [21]:
def build_dataset(cities, api_key):
    results = []

    for city in cities:
        print(f"Fetching data for {city}...")
        data = get_city_data(city, api_key)

        if data:
            results.append(data)

        # Prevent rate limiting
        time.sleep(1)

    df = pd.DataFrame(results)
    return df

In [22]:
df = build_dataset(cities, API_KEY)
df

Fetching data for Louisville...
Fetching data for Lexington...
Fetching data for Cincinnati...
Fetching data for Paducah...
Fetching data for Bowling Green...
Fetching data for Owensboro...
Fetching data for Elizabethtown...


,date_raw,date,city,region,country,temp_f,feels_like,condition,humidity,wind_mph,...,min_temp_day_1,avg_temp_day_1,date_2,max_temp_day_2,min_temp_day_2,avg_temp_day_2,date_3,max_temp_day_3,min_temp_day_3,avg_temp_day_3
0,2026-04-28,"Tuesday, April 28 2026",Louisville,Kentucky,United States of America,73.0,76.8,Sunny,66,3.8,...,61.3,67.9,2026-04-29,65.8,52.3,60.7,2026-04-30,63.7,43.3,54.0
1,2026-04-28,"Tuesday, April 28 2026",Lexington-Fayette,Kentucky,United States of America,73.0,76.5,Partly cloudy,64,7.2,...,59.4,67.9,2026-04-29,65.5,49.0,59.7,2026-04-30,60.0,39.3,49.8
2,2026-04-28,"Tuesday, April 28 2026",Cincinnati,Ohio,United States of America,71.1,71.1,Partly cloudy,53,6.3,...,58.3,64.0,2026-04-29,60.5,47.7,56.3,2026-04-30,59.5,40.8,50.2
3,2026-04-28,"Tuesday, April 28 2026",Paducah,Kentucky,United States of America,63.0,63.0,Patchy light rain with thunder,90,5.8,...,65.3,72.2,2026-04-29,67.3,50.7,61.7,2026-04-30,64.4,44.2,55.0
4,2026-04-28,"Tuesday, April 28 2026",Bowling Green,Kentucky,United States of America,75.9,78.1,Sunny,69,4.5,...,64.0,71.9,2026-04-29,66.3,51.2,61.5,2026-04-30,63.5,42.7,53.4
5,2026-04-28,"Tuesday, April 28 2026",Owensboro,Kentucky,United States of America,70.3,70.3,Light rain,78,2.2,...,62.2,69.7,2026-04-29,66.2,50.4,60.6,2026-04-30,63.8,43.3,54.3
6,2026-04-28,"Tuesday, April 28 2026",Elizabethtown,Kentucky,United States of America,73.2,76.9,Partly cloudy,78,2.2,...,63.9,71.0,2026-04-29,66.8,49.2,60.7,2026-04-30,62.5,41.4,52.6


In [18]:
df.to_csv("../data/API_weather_data.csv", index=False)